# Dimensiereductie

Auteurs: Brian van der Bijl (brian.vanderbijl@hu.nl), Tijmen Muller

- Studentnummer: 1853447 
- Naam: Dominik Enzinger
- Datum: 3/2/2026

## Deel I: Principal Component Analysis (PCA)

Het _Principal Component Analysis_ (PCA) algoritme kan gebruikt worden om het aantal dimensies van een dataset te reduceren tot de belangrijkste componenten. Als de originele dataset $n$ dimensies heeft, dan kunnen we met onderstaande stappen dit terugbrengen tot een (zelfgekozen) aantal van $n^\prime$ dimensies.

1. Centreer de data.
2. Bereken de covariantie van alle features onderling. 
3. Bereken de Eigenvectors en Eigenvalues van de covariantiematrix.
4. Kies de $n^\prime$ Eigenvectors om de dimensiereductie mee uit te voeren.
5. Vermenigvuldig de $n^\prime$ Eigenvectors met de originele data om de reductie toe te passen.

### Context

Gegeven is een databestand met embeddings van 200 tekstfragmenten. Elke embedding bestaat in 15 dimensies, en is gelabeled met een categorie. We gaan dimensionaliteitsreductie toepassen om de data te kunnen plotten.

De categorie geeft aan in welk genre de tekstfragmenten thuishoren. Daarnaast is onderscheid gemaakt tussen het perspectief waarin het fragment geschreven is: ik (1st person) of hij/haar/hen (3rd person):
- Fantasy (1st person)
- Fantasy (3rd person)
- Science Fiction (1st person)
- Science Fiction (3rd person)
- Romance (1st person)
- Romance (3rd person)
- Crime (1st person)
- Crime (3rd person)

In [1]:
import numpy as np
import pickle
import matplotlib.pyplot as plt

# Show floats on 3 digits, suppress scientific notation
np.set_printoptions(precision=3, suppress=True)

In [2]:
with open('data.pkl', 'rb') as file:
    data = pickle.load(file)

data.head(5)

e_1,e_2,e_3,e_4,e_5,e_6,e_7,e_8,e_9,e_10,e_11,e_12,e_13,e_14,e_15,label
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
1.108966,0.827991,1.627138,0.925778,0.945613,3.722442,0.457359,2.669865,0.805594,1.061672,0.762532,1.0722,0.539163,0.620247,0.380938,"""Fantasy (1st person)"""
1.653362,1.554929,1.475294,0.677498,1.302392,3.866261,0.804821,3.102261,1.228692,0.793275,0.338367,1.633079,1.049915,1.0656,0.299805,"""Fantasy (1st person)"""
1.225105,1.216584,1.423586,1.248953,0.965199,3.774443,0.491922,2.882402,1.151147,0.659875,0.285644,1.344977,1.0466,1.127903,0.689821,"""Fantasy (1st person)"""
1.732045,1.577317,1.555516,1.323991,1.093897,3.89912,0.153112,2.815527,1.318954,0.530422,0.866516,1.354425,0.503422,0.952766,0.185922,"""Fantasy (1st person)"""
1.2672,1.169387,0.949254,0.692989,1.048017,3.948163,0.512988,3.215607,1.02247,0.983851,0.886414,1.369123,0.383284,0.710478,0.572426,"""Fantasy (1st person)"""


### Voorbereidende opdracht

Gegeven een dataset met $m$ datapunten met elk $n$ features en een gewenste reductie tot $n^\prime$ dimensies. Bepaal voor elk van de vijf stappen van het algoritme wat de dimensies (oftewel `shape`) is van de volgende tussenresultaten:

1. De matrix met de originele dataset.
2. De matrix met de gemiddelden per feature om de data mee te centreren.
3. De covariantiematrix.
4. De matrix met de Eigenvectors en de matrix met de Eigenvalues.
5. De matrix met de _geselecteerde_ Eigenvectors.
6. De matrix met de _gereduceerde_ data.

**Antwoorden:**  

Ik heb een aantal prints in de PCA-functie gezet om te laten zien wat de dimensies van de matrices zijn tijdens het doorlopen van de stappen van PCA. Verder heb ik ook in de prints gezet waarvan elke dimensie afhangt (datapunten, features, n_components).

1. Het aantal dimensies van de originele dataset hangt af van het aantal datapunten en het aantal features; dus $D^{m \times n}$ of $D^{n \times m}$

2. Het aantal dimensies van de matrix met de gemiddelden per feature wordt bepaald door het aantal dimensies van de originele dateset. Om van elke waarde in de dataset het gemiddelde af te kunnen halen kunnen we een matrix gebruiken met het gemiddelde van elke feature (rij of kolom; afhankelijk van het format); dus $D^{m \times n}$ of $D^{n \times m}$

- 2,5. We zouden ook een vector kunnen gebruiken en de vector met shape $D^{n \times 1}$ broadcasten over de matrix. (Dit is ook wat numpy doet.)

3. Het aantal dimensies van de covariantiematrix is afhankelijk van het aantal features, omdat we elke feature met elke andere feature willen vergelijken; dus $D^{n \times n}$   

4. De matrix met de eigenvectors is even groot als de covariantiematrix, omdat de covariantiematrix symetrisch is zijn er het maximale aantal eigenvectors; dus $D^{n \times n}$  
De vector met de eigenvalues heeft het zelfde aantal dimensies als het aantal features; dus $D^{n \times 1}$

5. Het aantal dimensies van de geselcteerde eigenvectors hangt af van het aantal eigenvectors die behouden blijven en het aantal features; dus $D^{n' \times n}$ of $D^{n \times n'}$

6. Het aantal dimensies van de gereduceerde data is afhankelijk van het aantal gekozen eigenvectors en het aantal datapunten; dus $D^{m \times n'}$ of $D^{n' \times m}$


### Opdracht 1. Implementatie

Schrijf een eigen implementatie van het PCA-algoritme `compute_pca(X, n_components)` volgens eerdergenoemde stappen van het algoritme. Maak slim gebruik van fucties van `numpy` waar mogelijk, maar zorg wel dat je begrijpt wat je in elke stap doet. De laatste stap is al gegeven in de functiedefinitie hieronder.

Hint: Laat bij stap 3. zien (bijvoorbeeld met een `print()` statement) dat de meest informatieve Eigenvalue al meer dan 50% van de informatie bevat van onze dataset.

#### Input
- `X: numpy.array` - numpy matrix met dimensies $(m, n)$; elke rij is een datapunt in $n$ dimensies
- `n_components: int` - het gewenste aantal dimensies $n^\prime$

#### Output
`X_reduced: numpy.array` - een $(m, n^\prime)$ numpy matrix met de gereduceerde data.

In [ ]:
def compute_pca(X, n_components, prints=False):
    """
    Parameters
    ----------
    X : numpy.ndarray
        Input data matrix of shape (m, n), where m is the number of samples and n is the number of features.
    n_components : int
        The number of principal components (dimensions) to keep.
    prints : boolean
        Whether or not to print the pca process.

    Returns
    -------
    X_reduced : numpy.ndarray
        The data projected onto the top n_components principal components.
    """
    matrix = np.array(X)

    # Step 1: Center or normalize the matrix
    centered_data = matrix - matrix.mean(axis=0)  # axis 0 is the vertical axis, thus mean/ of each column.
    # normalized_data = (matrix - matrix.mean(axis=0)) / matrix.std(axis=0)  # We could also normalize the data

    # Step 2: Calculate the covariance matrix
    covariance_matrix = np.cov(centered_data, rowvar = False)

    # Step 3: Calculate eigenvalues and eigenvectors
    eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
    sorting_helper = np.argsort(eigenvalues)[::-1]
    sorted_eigenvalues = eigenvalues[sorting_helper]
    sorted_eigenvectors = eigenvectors[:,sorting_helper]
    eigenvalue_impact = sorted_eigenvalues / sum(sorted_eigenvalues)

    # Step 4: Select eigenvectors (pc's, principal components) to keep
    selected_eigenvectors = sorted_eigenvectors[:,:n_components]

    # Prints:
    if prints:
        print("Data matrix shape (datapoints, features):", matrix.shape, sep="\n", end="\n\n")
        print("Data matrix:", matrix, sep="\n", end="\n\n")
        print("Mean vector shape (features, 1):", matrix.mean(axis=0).shape, sep="\n", end="\n\n")
        print("Mean vector -> getting broadcast:", matrix.mean(axis=0), sep="\n", end="\n\n")
        print("Centered data shape (datapoints, features):", centered_data.shape, sep="\n", end="\n\n")
        print("Centered Data:", centered_data, sep="\n", end="\n\n")
        print("Covariance matrix shape (features, features):", covariance_matrix.shape, sep="\n", end="\n\n")
        print("Covariance matrix:", covariance_matrix, sep="\n", end="\n\n")
        print("Eigenvalues shape (features, 1):", eigenvalues.shape, sep="\n", end="\n\n")
        print("Eigenvalues:", eigenvalues, sep="\n", end="\n\n")
        print("Eigenvectors shape (features, features):", eigenvectors.shape, sep="\n", end="\n\n")
        print("Eigenvectors:", eigenvectors, sep="\n", end="\n\n")
        print("Sorting helper (indices):", sorting_helper, sep="\n", end="\n\n")
        print("Sorted eigenvalues shape (features, 1):", sorted_eigenvalues.shape, sep="\n", end="\n\n")
        print("Sorted eigenvalues:", sorted_eigenvalues, sep="\n", end="\n\n")
        print("Eigenvalue impact (percentage of impact):", eigenvalue_impact, sep="\n", end="\n\n")
        print("Sorted eigenvectors shape (features, features):", sorted_eigenvectors.shape, sep="\n", end="\n\n")
        print("Sorted eigenvectors:", sorted_eigenvectors, sep="\n", end="\n\n")
        print("Selected eigenvectors shape (features, n_components):", selected_eigenvectors.shape, sep="\n", end="\n\n")
        print("Selected eigenvectors:", selected_eigenvectors, sep="\n", end="\n\n")
        print("Result shape (datapoints, n_components):", (centered_data @ selected_eigenvectors).shape, sep="\n", end="\n\n")
        print("Result:", centered_data @ selected_eigenvectors, sep="\n", end="\n\n")

    # Stap 5: Multiply chosen eigenvectors with the centered or normalized dataset.
    return centered_data @ selected_eigenvectors

#### Test-scenario
Onderstaande code zou de volgende output moeten opleveren (het minteken kan wisselen):

```python
[[ 0.43437323 -0.49820384]
 [ 0.42077249  0.50351448]
 [-0.85514571 -0.00531064]]
 ```

In [4]:
np.random.seed(1)
X = np.random.rand(3, 10)
X_reduced = compute_pca(X, n_components=2, prints=True)

Data matrix shape (datapoints, features):
(3, 10)

Data matrix:
[[0.417 0.72  0.    0.302 0.147 0.092 0.186 0.346 0.397 0.539]
 [0.419 0.685 0.204 0.878 0.027 0.67  0.417 0.559 0.14  0.198]
 [0.801 0.968 0.313 0.692 0.876 0.895 0.085 0.039 0.17  0.878]]

Mean vector shape (features, 1):
(10,)

Mean vector -> getting broadcast:
[0.546 0.791 0.173 0.624 0.35  0.552 0.23  0.314 0.236 0.538]

Centered data shape (datapoints, features):
(3, 10)

Centered Data:
[[-0.129 -0.071 -0.173 -0.322 -0.203 -0.46  -0.043  0.031  0.161  0.   ]
 [-0.126 -0.106  0.032  0.254 -0.323  0.118  0.188  0.244 -0.095 -0.34 ]
 [ 0.255  0.177  0.141  0.068  0.526  0.342 -0.144 -0.275 -0.066  0.34 ]]

Covariance matrix shape (features, features):
(10, 10)

Covariance matrix:
[[ 0.049  0.034  0.027  0.013  0.101  0.066 -0.028 -0.053 -0.013  0.065]
 [ 0.034  0.024  0.017  0.004  0.071  0.04  -0.021 -0.038 -0.006  0.048]
 [ 0.027  0.017  0.025  0.037  0.049  0.066 -0.003 -0.018 -0.02   0.018]
 [ 0.013  0.004  0.037  0

### Opdracht 2. Visualisatie met dimensiereductie

Maak op basis van de aangeleverde `data` een numpy array van de datapunten, en gebruik je PCA-implementatie om een 2D- en 3D-weergave van de data te maken. Maak van elke weergave een plot, waarbij iedere categorie een eigen kleur krijgt.

In [5]:
# TODO: Schrijf hier je code.

In [6]:
# TODO: Schrijf hier je code.

### Opdracht 3. Analyse

Analyseer de resultaten:
1. Welke categorieën zijn op basis van de PCA-reductie te onderscheiden, en welke niet? 
2. Geef aan hoeveel procent van de informatie bewaard is gebleven in 2D en 3D respectievelijk.
3. Hoeveel dimensies zijn nodig om 90% van de informatie te bewaren?
4. En voor 95%?

_Schrijf hier je antwoord._

## Deel II: t-Distributed Stochastic Neighbour Embedding (t-SNE)

Een alternatieve methode voor dimensiereductie is _t-Distributed Stochastic Neighbour Embedding_ (t-SNE). 

### Opdracht 4. Toepassing
Gebruik SciKit-Learn om met behulp van t-SNE de data tot 2 dimensies te reduceren en plot het resultaat (wederom met kleuren voor de categorien). 

In [7]:
# TODO: Schrijf hier je code.

### Opdracht 5. Vergelijking

Vergelijk deze met de resultaten van je PCA-implementatie:

1. Hoe verhoudt de zichtbaarheid van de categorieën zich tussen beide resultaten?
2. Hoe verhouden de algoritmes zich in het behoud van informatie?

Beantwoord de volgende vragen los van de data van deze opdracht:

3. Waarvoor zou je PCA en t-SNE inzetten als je te maken krijgt met een onbekende (mogelijk ongelabelde) dataset?
4. Geef een voorbeeld waar PCA de voorkeur heeft boven t-SNE.
5. Geef een voorbeeld waar t-SNE de voorkeur heeft boven PCA.

_Schrijf hier je antwoord._

### Opdracht 6. Project

Als het goed is, heb je op dit moment een eerste idee van de data waar je in het project mee gaat werken. Geef antwoord op onderstaande vragen.

1. Wat is de dimensionaliteit waar je mee te maken hebt?
2. Beschrijf hoe dimensionaliteitsreductie-algoritmen je kunnen helpen de data te verkennen.

_Schrijf hier je antwoord._